In [1]:
import pandas as pd
import numpy as np
df = pd.read_csv(
"feature_engineered.csv"
)

In [2]:
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

seg_cols = [

'Customer_Activity_Score',

'Loyalty_Score',

'Lifetime_Value',

'Total_Purchases',

'Average_Order_Value',

'Customer_Friction_Index'

]

scaler = StandardScaler()

X_seg = scaler.fit_transform(
df[seg_cols]
)

kmeans = KMeans(
n_clusters=5,
random_state=42,
n_init=10
)

df["Cluster"] = kmeans.fit_predict(
X_seg
)

In [3]:
np.random.seed(42)

df["Experiment_Group"] = np.random.choice(
    ["Control","Treatment"],
    size=len(df)
)

In [4]:
df["Retained"] = 1-df["Churned"]

In [5]:
treatment_mask = (
    df["Experiment_Group"]
    == "Treatment"
)

boost = np.random.binomial(
    1,
    0.05,
    len(df)
)

df["Experiment_Retained"] = df["Retained"]

df.loc[
    treatment_mask,
    "Experiment_Retained"
] = np.minimum(
    1,
    df.loc[
        treatment_mask,
        "Retained"
    ] + boost[
        treatment_mask
    ]
)

In [6]:
experiment = df.groupby(
"Experiment_Group"
)["Experiment_Retained"].mean()

experiment

Experiment_Group
Control      0.712210
Treatment    0.723732
Name: Experiment_Retained, dtype: float64

In [7]:
from statsmodels.stats.proportion import proportions_ztest

control = df[
    df["Experiment_Group"]=="Control"
]

treatment = df[
    df["Experiment_Group"]=="Treatment"
]

success = [control[
"Experiment_Retained"
].sum(),

treatment[
"Experiment_Retained"
].sum()]

nobs = [len(control),len(treatment)]

z,p = proportions_ztest(
success,
nobs
)

print("Z-score:",z)

print("P-value:",p)

Z-score: -2.8627381984845486
P-value: 0.004199974352964506


In [8]:
if p<0.05:

    print("Statistically significant → Deploy treatment")

else:

    print("Not statistically significant → More testing needed")

Statistically significant → Deploy treatment


In [9]:

df["Experiment_Revenue"] = df["Lifetime_Value"]

treatment_mask = (
    df["Experiment_Group"]
    == "Treatment"
)

revenue_boost = np.random.normal(
    50,
    20,
    len(df)
)

df.loc[
treatment_mask,
"Experiment_Revenue"
] += revenue_boost[
treatment_mask
]

In [10]:
df["Experiment_Cart_Abandonment"] = (
df["Cart_Abandonment_Rate"]
)

cart_improvement = np.random.normal(
3,
1,
len(df)
)

df.loc[
treatment_mask,
"Experiment_Cart_Abandonment"
] -= cart_improvement[
treatment_mask
]

df[
"Experiment_Cart_Abandonment"
] = np.maximum(
0,
df[
"Experiment_Cart_Abandonment"
]
)

In [11]:
df[
"Experiment_Activity"
] = df[
"Customer_Activity_Score"
]

activity_boost = np.random.normal(
0.03,
0.01,
len(df)
)

df.loc[
treatment_mask,
"Experiment_Activity"
] += activity_boost[
treatment_mask
]

In [12]:
metrics = [

"Experiment_Retained",

"Experiment_Revenue",

"Experiment_Cart_Abandonment",

"Experiment_Activity"

]

experiment_summary = (

df.groupby(
"Experiment_Group"
)[metrics]

.mean()

)

experiment_summary

,Experiment_Retained,Experiment_Revenue,Experiment_Cart_Abandonment,Experiment_Activity
Experiment_Group,,,,
Control,0.712210,1441.649384,57.126155,0.296362
Treatment,0.723732,1489.707091,54.033932,0.326667


In [13]:
control = experiment_summary.loc[
"Control"
]

treatment = experiment_summary.loc[
"Treatment"
]

uplift = (

(treatment-control)

/

control

)*100

uplift

Experiment_Retained             1.617801
Experiment_Revenue              3.333523
Experiment_Cart_Abandonment    -5.412972
Experiment_Activity            10.225687
dtype: float64

In [14]:
cluster_results = []

for cluster in sorted(
df["Cluster"].unique()
):

    temp = df[
        df["Cluster"]==cluster
    ]

    grp = temp.groupby(
        "Experiment_Group"
    )["Experiment_Retained"].mean()

    control = grp["Control"]

    treatment = grp["Treatment"]

    uplift = (
        (
        treatment-control
        )

        /

        control

    )*100

    cluster_results.append(

        {

        "Cluster":cluster,

        "Control":

        control,

        "Treatment":

        treatment,

        "Uplift_%":

        uplift

        }

    )

cluster_df = pd.DataFrame(
cluster_results
)

cluster_df

,Cluster,Control,Treatment,Uplift_%
0,0,0.814533,0.805858,-1.065079
1,1,0.795470,0.803015,0.948526
2,2,0.714286,0.636364,-10.909091
3,3,0.529502,0.549986,3.868684
4,4,0.762165,0.782615,2.683251


In [15]:
from statsmodels.stats.proportion import proportions_ztest

cluster_significance=[]

for cluster in sorted(
df["Cluster"].unique()
):

    temp = df[
    df["Cluster"]==cluster
    ]

    control = temp[
    temp[
    "Experiment_Group"
    ]=="Control"
    ]

    treatment = temp[
    temp[
    "Experiment_Group"
    ]=="Treatment"
    ]

    success = [

    control[
    "Experiment_Retained"
    ].sum(),

    treatment[
    "Experiment_Retained"
    ].sum()

    ]

    nobs = [

    len(control),

    len(treatment)

    ]

    z,p = proportions_ztest(
    success,
    nobs
    )

    cluster_significance.append(

    {

    "Cluster":cluster,

    "P_Value":p

    }

    )

pd.DataFrame(
cluster_significance
)

,Cluster,P_Value
0,0,0.445536
1,1,0.208065
2,2,0.678440
3,3,0.013264
4,4,0.005758


In [16]:
recommendations=[]

for idx,row in df.iterrows():

    rec=[]

    if row["Cluster"]==3:
        rec.append(
        "High churn risk: retention package"
        )

    if row["Customer_Friction_Index"]>12:
        rec.append(
        "Priority support routing"
        )

    if row["Cart_Abandonment_Rate"]>60:
        rec.append(
        "Cart reminder intervention"
        )

    if row["Lifetime_Value"]>2500:

        rec.append(
        "VIP retention strategy"
        )

    recommendations.append(
        "; ".join(rec)
    )

df[
"Recommendation"
]=recommendations

In [17]:
df["Recommendation"].value_counts().head(15)

Recommendation
                                                                                                                    16645
High churn risk: retention package; Priority support routing; Cart reminder intervention                            10708
Priority support routing; Cart reminder intervention                                                                 4866
VIP retention strategy                                                                                               4636
Cart reminder intervention                                                                                           4621
Priority support routing                                                                                             3429
High churn risk: retention package; Cart reminder intervention                                                       1611
High churn risk: retention package; Priority support routing                                                         1536
High chur

In [18]:
df[
[
"Cluster",
"Customer_Friction_Index",
"Cart_Abandonment_Rate",
"Lifetime_Value",
"Recommendation"
]
].sample(10)

,Cluster,Customer_Friction_Index,Cart_Abandonment_Rate,Lifetime_Value,Recommendation
36338,1,9.64,56.4,1299.09,
27324,4,7.19,41.9,3020.96,VIP retention strategy
26135,1,12.38,73.8,944.25,Priority support routing; Cart reminder interv...
11630,4,4.69,26.9,1396.96,
45646,1,9.10,51.0,1909.72,
35958,3,17.16,71.6,899.03,High churn risk: retention package; Priority s...
35400,1,7.65,56.5,916.23,
11363,3,10.95,69.5,872.13,High churn risk: retention package; Cart remin...
22491,3,14.91,59.1,731.29,High churn risk: retention package; Priority s...
44653,0,9.42,34.2,2004.63,


In [19]:
df.to_csv(
    "../data/experimentation_data.csv",
    index=False
)